# 1. Identifying halo-orbit station-keeping dynamics

**Question:** Can a phase-dependent state-space model predict the response to a small impulse, and use that prediction to choose its magnitude and orbital phase so that position error stays below **10 km** and velocity error below **0.1 m/s**?

A near-rectilinear halo orbit (**NRHO**) is a repeating path near the Moon. Small departures can grow without corrective velocity changes, called **impulses** or $\Delta v$. We study only the observer's own orbit. Phase is elapsed time through one period, expressed from 0° to 360°; it is not a geometric angle.

The identification plant maps three commanded impulse components $u_k$ [m/s] to six position/velocity departure components $y_k$. The guidance calculation later uses this plant in the reverse direction to choose a burn. These are separate tasks.

In [ ]:
from pathlib import Path
import json
import numpy as np
from IPython.display import display, Markdown
from halo_tracking import stationkeeping as sk
from halo_tracking import stationkeeping_diagnostics as sd
from halo_tracking import stationkeeping_plots as plots
ROOT = Path.cwd()
if not (ROOT/'configs').exists():
    ROOT = ROOT.parent
OUT = ROOT/'artifacts'/'stationkeeping'
plots.style()
def table(headers, rows):
    display(Markdown('| '+' | '.join(headers)+' |\n| '+' | '.join(['---']*len(headers))+' |\n'+
                     '\n'.join('| '+' | '.join(map(str,row))+' |' for row in rows)))

orbit, data = sk.prepare_study(ROOT, OUT/'data')
training = data['training']
table(['Split','Independent trajectories','Seed range'],
      [(name,len(rec.seeds),f'{rec.seeds[0]}–{rec.seeds[-1]}') for name,rec in data.items()])
print(f'Uniform sampling: {orbit.dt_s/3600:.3f} h; nominal period: {orbit.T*orbit.cfg["time_unit_s"]/86400:.3f} day')
print('Open-loop random test impulses; independent synthetic full-state navigation measurements.')

## Input and output, by direction

$x$ points along the Earth–Moon line, $y$ lies in the orbital plane, and $z$ points out of that plane. Position errors use rotating axes. Velocity errors are **inertial velocity differences resolved on those same axes**, including the frame's transport term. Each displayed output is measured **before** its indexed impulse; $u_k$ affects $y_{k+1}$. The staircase shows impulse samples, not a continuously applied acceleration.

These are simulated identification experiments, not flight telemetry or an optical navigation demonstration. Small independent random commands provide excitation in all three directions. Whole trajectories have separate training, validation and test seeds; test data are reserved for Notebook 2.

In [ ]:
plots.input_output(orbit, training, OUT/'figures')

## Welch spectra and empirical frequency response

Following Studios 7–8, start with a Hann window, 50% overlap and 32 samples per segment, then repeat with 16, 64 and 128 to expose the resolution/variance trade. A 32-sample segment spans only a quarter orbit and cannot resolve the orbital fundamental. A full-orbit window improves resolution but leaves only one window per independent trajectory. Average spectra **within** separate trajectories before forming ratios. Never join endpoints from different experiments.

The full three-input estimate is $\hat H=\hat S_{yu}\hat S_{uu}^{-1}$; the cross-spectrum uses $Y\,U^*$. Pairwise coherence screens weak channels, while input spectral conditioning screens unreliable inversion. Because an orbit can be phase dependent and departures can be nonstationary, these are **finite-record summaries**, not an established global LTI transfer function. Input PSD bands use the conservative non-overlapping segment count; formal output/FRF confidence bands are deferred until stationarity is justified.

In [ ]:
s = orbit.settings
spec = sd.welch_bundle(training.u, training.y[:,:-1], 1/orbit.dt_s,
                       s['welch_length'], s['welch_overlap'])
table(['Length','Resolution [cycle/day]','Overlapping windows','Conservative DOF'],
      [(L, f'{86400/(orbit.dt_s*L):.3f}',
        sd.welch_bundle(training.u,training.y[:,:-1],1/orbit.dt_s,L)['segments'],
        sd.welch_bundle(training.u,training.y[:,:-1],1/orbit.dt_s,L)['dof']) for L in (16,32,64,128)])
plots.spectra(spec, orbit, OUT/'figures')
plots.frf(spec, OUT/'figures')

In [ ]:
import matplotlib.pyplot as plt
fig, axs = plt.subplots(1,2,figsize=(15,5))
for L in (16,32,64,128):
    q = sd.welch_bundle(training.u,training.y[:,:-1],1/orbit.dt_s,L)
    axs[0].semilogy(q['f'][1:]*86400,q['syy'][1:,0]/86400,label=f'L={L}')
    axs[1].semilogy(q['f'][1:]*86400,abs(q['H'][1:,0,0]),label=f'L={L}')
axs[0].set(xlabel='Frequency [cycle/day]',ylabel='x position PSD [km²/(cycle/day)]')
axs[1].set(xlabel='Frequency [cycle/day]',ylabel='Δv x → x position gain [km/(m/s)]')
for ax in axs:
    ax.grid(); ax.legend()
plots.finish(fig,OUT/'figures','02b_welch_resolution_sensitivity')

## What structure do the data support?

Studio 6 compares segment means, variances and autocovariances, then pairs ADF (unit-root null) with KPSS (stationarity null). Here the record is short and periodically varying dynamics can change its statistics. ADF/KPSS are screens with limited power; even agreement does not establish wide-sense stationarity. Shaded segment bands account approximately for correlation and are diagnostic, not simultaneous confidence regions.

In [ ]:
stationarity_rows = [sd.stationarity(training.u[0,:,c],'u '+ 'xyz'[c]) for c in range(3)]
stationarity_rows += [sd.stationarity(training.y[0,:,c],plots.NAMES[c]) for c in range(6)]
table(['Channel','ADF p','KPSS p*','Mean flags / 4','Variance flags / 4','Joint reading'],
      [(r['channel'],f"{r['adf_p']:.3g}",f"{r['kpss_p']:.3g}",r['mean_flags'],r['variance_flags'],r['reading'])
       for r in stationarity_rows])
print('* KPSS values at 0.01/0.10 can be table bounds; exact warnings are saved in stationarity.json.')
(OUT/'stationarity.json').write_text(json.dumps(stationarity_rows,indent=2)+'\n')
plots.stationarity_panels(training,orbit,OUT/'figures')

The next plots examine memory (input–output correlations), excitation rank, coupling across directions, and sensitivity to **orbital phase**. Compare nonlinear propagation with the linear state transition matrix (**STM**) across perturbation sizes: this tests the range over which linearization is useful. It does not prove global linearity. Unlike a damped laboratory structure, an uncontrolled halo orbit need not have stable LTI poles.

In [ ]:
diagnostics = sk.finite_horizon_diagnostics(orbit)
ccf = sd.lag_response(training.u,training.y[:,:-1],16)
plots.structure(orbit,training,diagnostics,ccf,spec,OUT/'figures')
np.savez_compressed(OUT/'structure_diagnostics.npz',**diagnostics)
print('Largest one-orbit Floquet multiplier magnitude:',float(abs(diagnostics['multipliers']).max()))
print('Reference closure [position km; velocity m/s]:',orbit.closure)
table(['Property','Interpretation'],[
    ('Plant','Nonlinear autonomous Earth–Moon CR3BP, with discrete impulses'),
    ('Departure model','Locally linear, periodically time varying about the nominal orbit'),
    ('Input/output','3 commanded impulses → 6 observed state departures; coupled MIMO'),
    ('Measurements','Full-state synthetic navigation; C=I, not an optical observability claim'),
    ('Identification','Open loop; known commands independent of navigation noise'),
    ('Guidance','Closed loop; phase and impulse selected from current estimated state')])

## Why station keeping is needed

The nominal orbit is numerically corrected to close before using its STM. Floquet multipliers describe growth over one orbit; a multiplier outside the unit circle indicates local linear instability. The following comparison propagates one small initial error continuously, with **no physical resets** and no burns. The dashed lines mark the required corridor.

In [ ]:
no_burn_day,no_burn_output = sk.uncontrolled(orbit)
plots.geometry(orbit,no_burn_day,no_burn_output,OUT/'figures')
np.savez_compressed(OUT/'uncontrolled.npz',day=no_burn_day,output=no_burn_output)

## Working choice for Notebook 2

Start with the six-state, phase-dependent STM grey box and estimate three actuator gains. It retains known cross-axis dynamics while allowing a measurable physical mismatch. The phase-sensitivity, coupling and linearization plots determine whether a frozen/local LTI approximation is credible; the full-record Welch peaks alone cannot select an LTI order.

Possible later comparisons are local phase-conditioned OE/BJ models or a periodic parameter basis (Studio 13). That decision remains open until these diagnostics are reviewed. **NARX is deferred.** See `docs/course_alignment.md` for method provenance and experiment limits.